# Phase 5 — RAG + LLM Explainability Layer

**Input files (from previous notebooks):**
- `data/contracts_ie_clean.csv` — cleaned contracts + all labels (NB01 + NB02a)
- `data/anomaly_labels.csv` — IsolationForest scores (NB03)
- `data/metrics_summary.csv` — model benchmark panel (NB03)

**Outputs saved to `data/`:**
- `ragas_results.json` — faithfulness, relevance, context precision scores
- `rag_audit_samples.json` — structured JSON audit reports for sample contracts

**Also produced:** `analyze_contract.py` — a CLI tool to run the RAG pipeline on any contract ID

---

## Why RAG and not plain LLM prompting?

A plain prompt like *"Is this contract suspicious?"* fed to an LLM produces confident-sounding
explanations that may be entirely fabricated — the model has no access to actual EU procurement law.

RAG (Retrieval-Augmented Generation) solves this by:
1. **Chunking** the actual legal text (EU Directive 2014/24/EU, OLAF guidelines) into a vector index
2. **Retrieving** the most relevant legal clauses for each contract at inference time
3. **Grounding** the LLM's explanation in those retrieved clauses

Every claim in the audit report is traceable to a specific article of EU law.

## Pipeline flow
```
Flagged Contract Notice
        ↓
Dense Retrieval (FAISS + sentence-transformers)
        ↓
Top-5 Legal Context Chunks
        ↓
Context-Augmented Prompt
        ↓
LLM (GPT-4o / Claude / Mistral)
        ↓
Structured JSON Audit Report
  { risk_score, risk_factors, regulation_references, explanation }
```

## 0. Colab / Local Setup

In [ ]:
import sys, subprocess, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(['pip', 'install', '-q',
        'langchain', 'langchain-community', 'langchain-openai', 'langchain-ollama',
        'faiss-cpu', 'sentence-transformers',
        'pydantic', 'ragas', 'datasets', 'tqdm'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/fraud_detection_nlp/data/'
else:
    DATA_DIR = '../data'

os.makedirs(DATA_DIR, exist_ok=True)
print(f'DATA_DIR: {DATA_DIR}')


## 1. Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import List, Optional

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

warnings.filterwarnings('ignore')
print('Imports OK')

## 2. LLM Configuration

Set your API key here. The pipeline is model-agnostic —
swap `ChatOpenAI` for `ChatAnthropic` or a local `Ollama` model
by changing just this cell.

In [ ]:
# ─── LLM Configuration ────────────────────────────────────────────────────────
# Priority:
#   1. Ollama (local, free, no API key)  ← recommended for school use
#   2. OpenAI (requires OPENAI_API_KEY)
#   3. Rule-based fallback (no LLM at all)
#
# To use Ollama:
#   1. Install from https://ollama.com
#   2. Run: ollama pull mistral
#   3. Run: ollama serve
#   Then set LLM_BACKEND = 'ollama' below.

LLM_BACKEND = os.getenv('LLM_BACKEND', 'ollama')  # 'ollama' | 'openai' | 'rule'
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'mistral')

llm = None

if LLM_BACKEND == 'ollama':
    try:
        from langchain_community.llms import Ollama
        llm = Ollama(model=OLLAMA_MODEL, temperature=0)
        print(f'LLM: Ollama ({OLLAMA_MODEL}) — local, no API key needed')
    except Exception as e:
        print(f'Ollama not available ({e}). Falling back to rule-based mode.')
        LLM_BACKEND = 'rule'

elif LLM_BACKEND == 'openai':
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
    if not OPENAI_API_KEY:
        print('WARNING: OPENAI_API_KEY not set. Falling back to rule-based mode.')
        LLM_BACKEND = 'rule'
    else:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=OPENAI_API_KEY)
        print(f'LLM: OpenAI gpt-4o-mini')

if LLM_BACKEND == 'rule':
    print('LLM: rule-based fallback (no LLM — structured reports generated from flag logic)')

# Embedding model — same as NB02a for consistency
EMBED_MODEL = 'paraphrase-multilingual-mpnet-base-v2'
from langchain_community.embeddings import HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
print(f'Embedding model: {EMBED_MODEL}')


## 3. Load Contract Data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'contracts_ie_clean.csv'), low_memory=False)
anomaly = pd.read_csv(os.path.join(DATA_DIR, 'anomaly_labels.csv'))

df = df.merge(anomaly[['contract_id', 'iso_forest_flag', 'iso_forest_score']],
              on='contract_id', how='left')

print(f'Contracts loaded: {len(df):,}')

# Suspicious contracts: flagged by at least one supervised or unsupervised model
df['any_flag'] = (
    df['winner_concentration'].eq(1) |
    df['copy_paste_description'].eq(1) |
    df['iso_forest_flag'].eq(1)
).astype(int)

suspicious = df[df['any_flag'] == 1].copy()
print(f'Suspicious contracts (any flag): {len(suspicious):,}')

## 4. Legal Document Corpus

We build the RAG knowledge base from three sources:
1. **EU Directive 2014/24/EU** — the primary public procurement law
2. **OLAF Anti-Fraud Guidelines** — European Anti-Fraud Office red flag definitions
3. **OpenTender Fraud Typologies** — the definitions behind the indicators we used

### Chunking strategy
- **Chunk size: 400 tokens** — large enough to contain a full legal clause with context
- **Overlap: 50 tokens** — prevents a sentence from being split across two chunks
  with no shared context

Smaller chunks (100–200 tokens) give finer retrieval but lose clause context.
Larger chunks (800+ tokens) reduce retrieval precision. 400 is the standard
recommendation for legal/regulatory text.

In [ ]:
# Legal reference text
# In production: load from PDF / TXT files.
# Here we include the most relevant articles directly as strings
# so the notebook runs without external file dependencies.

LEGAL_CORPUS = [
    {
        "source": "EU Directive 2014/24/EU — Article 18: Principles of procurement",
        "text": (
            "Contracting authorities shall treat economic operators equally and without discrimination "
            "and shall act in a transparent and proportionate manner. The design of the procurement "
            "shall not be made with the intention of excluding it from the scope of this Directive or "
            "of artificially narrowing competition. Competition shall be considered to be artificially "
            "narrowed where the design of the procurement is made with the intention of unduly "
            "favouring or disadvantaging certain economic operators."
        )
    },
    {
        "source": "EU Directive 2014/24/EU — Article 26: Choice of procedures",
        "text": (
            "When awarding public contracts, contracting authorities shall apply the national procedures "
            "adjusted in conformity with this Directive, provided that a prior call for competition has "
            "been published, subject to this Directive. Contracting authorities may apply the negotiated "
            "procedure without prior publication only in specific, strictly limited circumstances. "
            "Unjustified use of negotiated procedure without prior publication is a procurement irregularity."
        )
    },
    {
        "source": "EU Directive 2014/24/EU — Article 27: Open procedure (time limits)",
        "text": (
            "In open procedures, the time limit for receipt of tenders shall be at least 35 days from "
            "the date on which the contract notice was sent for publication. Where contracting authorities "
            "have published a prior information notice, they may reduce the minimum time limit for receipt "
            "of tenders to 15 days. A time limit shorter than 35 days that is not justified by urgency "
            "constitutes an abnormally short tender period and is a transparency and integrity red flag."
        )
    },
    {
        "source": "EU Directive 2014/24/EU — Article 46: Division of contracts into lots",
        "text": (
            "Contracting authorities may decide to award a contract in the form of separate lots and may "
            "determine the size and subject-matter of such lots. Contracting authorities shall indicate "
            "the main reasons for their decision not to subdivide into lots. Artificially splitting a "
            "contract into smaller lots to avoid publication thresholds constitutes contract splitting "
            "and is prohibited under EU procurement law."
        )
    },
    {
        "source": "OLAF Anti-Fraud Guidelines — Single Bidding",
        "text": (
            "Single bidding occurs when only one tender is submitted in response to a competitive "
            "procurement procedure. While not illegal per se, a pattern of single bidding by the same "
            "supplier to the same buyer is a significant red flag for bid rigging, collusion, or "
            "specifications tailored to favour a specific supplier. OLAF recommends investigation when "
            "the single-bid rate exceeds 30% for a buyer-supplier pair over a 3-year period."
        )
    },
    {
        "source": "OLAF Anti-Fraud Guidelines — Winner Concentration",
        "text": (
            "Winner concentration refers to the repeated award of contracts by the same contracting "
            "authority to the same economic operator. A supplier winning more than 50% of contracts "
            "from a single buyer over a multi-year period is a structural red flag for collusion, "
            "corruption, or market rigging. OLAF recommends cross-checking company registration data, "
            "beneficial ownership records, and whether competing bidders share addresses or ownership."
        )
    },
    {
        "source": "OLAF Anti-Fraud Guidelines — Copy-Paste Specifications",
        "text": (
            "Identical or near-identical technical specifications across multiple contracting authorities "
            "may indicate that a supplier has pre-written the specification to ensure a sole-source award. "
            "This is particularly suspicious when the specifications are issued by legally independent "
            "buyers with no shared framework agreement. Text similarity above 85-90% across different "
            "buyers with no documented coordination constitutes a copy-paste red flag."
        )
    },
    {
        "source": "OpenTender Fraud Typology — INTEGRITY_SINGLE_BID indicator",
        "text": (
            "The INTEGRITY_SINGLE_BID indicator is set to 100 when exactly one bid is received for a "
            "competitive procurement lot. It is set to 0 when two or more bids are received. "
            "INSUFFICIENT_DATA is returned when the number of bids is not reported in the contract "
            "award notice. A value of 100 on this indicator does not prove fraud, but triggers "
            "further review under OLAF guidelines."
        )
    },
    {
        "source": "OpenTender Fraud Typology — INTEGRITY_ADVERTISEMENT_PERIOD indicator",
        "text": (
            "The INTEGRITY_ADVERTISEMENT_PERIOD indicator measures whether the time between publication "
            "of the contract notice and the bid deadline meets minimum statutory requirements. "
            "A value of 100 indicates the advertisement period is shorter than the minimum defined "
            "by EU Directive 2014/24/EU Article 27 (35 days for open procedures, 30 days for restricted). "
            "Urgency derogations must be explicitly stated and documented in the award notice."
        )
    },
]

print(f'Legal corpus: {len(LEGAL_CORPUS)} documents loaded')

## 5. Build FAISS Vector Index

In [ ]:
FAISS_PATH = os.path.join(DATA_DIR, 'faiss_legal_index')

# Text splitter — 400 token chunks, 50 token overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ']
)

if os.path.exists(FAISS_PATH):
    print('Loading existing FAISS index from disk...')
    vectorstore = FAISS.load_local(
        FAISS_PATH, embedder, allow_dangerous_deserialization=True
    )
    print('FAISS index loaded.')
else:
    print('Building FAISS index...')

    # Convert corpus entries to LangChain Document objects
    docs = []
    for entry in LEGAL_CORPUS:
        chunks = splitter.split_text(entry['text'])
        for chunk in chunks:
            docs.append(Document(
                page_content=chunk,
                metadata={'source': entry['source']}
            ))

    print(f'Total chunks: {len(docs)}')

    vectorstore = FAISS.from_documents(docs, embedder)
    vectorstore.save_local(FAISS_PATH)
    print(f'FAISS index built and saved to: {FAISS_PATH}')

## 6. Structured Output Schema

We enforce a fixed JSON schema for every audit report using Pydantic.
This guarantees the LLM cannot return free-form text — every field
is validated before being stored.

**Why structured output?**
- Downstream systems (dashboards, case management tools) need predictable fields
- It prevents the model from padding the answer with disclaimers
- It forces the model to commit to a numeric risk score rather than vague language

In [ ]:
class AuditReport(BaseModel):
    """Structured audit report for a single suspicious contract."""

    contract_id: str = Field(description="The persistent contract identifier")
    risk_score: int = Field(
        ge=0, le=100,
        description="Overall risk score from 0 (no risk) to 100 (highest risk)"
    )
    risk_factors: List[str] = Field(
        description="List of specific fraud indicators triggered by this contract"
    )
    regulation_references: List[str] = Field(
        description="List of EU Directive articles or OLAF guidelines that apply"
    )
    explanation: str = Field(
        description="Plain-English explanation of why this contract is suspicious, "
                    "grounded in the retrieved legal context"
    )
    recommended_action: str = Field(
        description="Recommended next step: MONITOR / INVESTIGATE / ESCALATE"
    )

print('AuditReport schema defined.')

## 7. RAG Pipeline

The pipeline for each contract:
1. Build a query string from the contract's flags and description
2. Retrieve the top-5 most relevant legal chunks from FAISS
3. Inject the retrieved context + contract data into a structured prompt
4. Call the LLM and parse the JSON output into an `AuditReport`

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

SYSTEM_PROMPT = """
You are an expert EU public procurement auditor.
Your task is to analyse a flagged contract notice and produce a structured audit report.

You must base your explanation ONLY on the legal context provided below.
Do not invent regulations or cite articles not present in the context.

--- LEGAL CONTEXT ---
{context}
--- END OF LEGAL CONTEXT ---

Respond with a valid JSON object matching this schema exactly:
{{
  "contract_id": string,
  "risk_score": integer 0-100,
  "risk_factors": [list of strings],
  "regulation_references": [list of strings],
  "explanation": string,
  "recommended_action": "MONITOR" | "INVESTIGATE" | "ESCALATE"
}}
"""

USER_PROMPT = """
Contract ID    : {contract_id}
Buyer          : {buyer_name}
Title          : {title}
Description    : {description}

Fraud flags triggered:
  - winner_concentration     : {winner_concentration}
  - copy_paste_description   : {copy_paste_description}
  - iso_forest_flag          : {iso_forest_flag}
  - iso_forest_score         : {iso_forest_score}
  - buyer_contracts_count    : {buyer_contracts_count}

Produce the audit report as a JSON object.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human',  USER_PROMPT)
])

print('RAG pipeline components ready.')

In [ ]:
# Rule-based fallback (used when LLM_BACKEND == 'rule' or LLM is unavailable)
_FLAG_RULES_NB04 = {
    "winner_concentration"  : ("Buyer in top 5% by contract volume.",
                                ["EU Directive 2014/24/EU — Art. 18"], 35),
    "copy_paste_description": ("Description near-identical to a different buyer's notice.",
                                ["EU Directive 2014/24/EU — Art. 18",
                                 "OLAF Guidelines — Copy-paste notices"], 30),
    "single_bid"            : ("Single tender received — insufficient competition.",
                                ["EU Directive 2014/24/EU — Art. 18",
                                 "OLAF Guidelines — Single bidding"], 25),
    "short_tender_period"   : ("Tender period below 35-day minimum (Art. 27).",
                                ["EU Directive 2014/24/EU — Art. 27"], 20),
}

def _rule_based_report(row: pd.Series) -> dict:
    score, factors, refs = 0, [], []
    for col, (factor, articles, weight) in _FLAG_RULES_NB04.items():
        if row.get(col) == 1:
            score += weight; factors.append(factor); refs.extend(articles)
    score  = min(score, 97)
    refs   = list(dict.fromkeys(refs))
    action = "ESCALATE" if score >= 65 else ("INVESTIGATE" if score >= 35 else "MONITOR")
    return {
        "contract_id"           : str(row.get("contract_id", "")),
        "risk_score"            : score,
        "risk_factors"          : factors,
        "regulation_references" : refs,
        "explanation"           : (
            f"Contract '{str(row.get('title',''))[:80]}' by {row.get('buyer_name','Unknown')} "
            f"triggered {len(factors)} flag(s). Risk score: {score}/100. "
            f"Recommended action: {action}."
        ),
        "recommended_action"    : action,
    }


def run_rag_audit(row: pd.Series) -> dict:
    """
    Run the full RAG pipeline for one contract row.
    Falls back to rule-based report if LLM is unavailable.
    Returns a dict matching the AuditReport schema, or an error dict.
    """
    if LLM_BACKEND == 'rule' or llm is None:
        return _rule_based_report(row)

    # 1. Build retrieval query
    query = (
        f"{row.get('title', '')}. "
        f"Flags: winner_concentration={row.get('winner_concentration', 0)}, "
        f"copy_paste={row.get('copy_paste_description', 0)}, "
        f"short_tender={row.get('short_tender_period', 'unknown')}, "
        f"single_bid={row.get('single_bid', 'unknown')}."
    )

    # 2. Retrieve top-5 legal chunks
    retrieved_docs = retriever.invoke(query)
    context = '\n\n'.join(
        f"[{doc.metadata['source']}]\n{doc.page_content}"
        for doc in retrieved_docs
    )

    # 3. Call LLM
    desc = str(row.get('description', ''))[:800]

    try:
        if LLM_BACKEND == 'ollama':
            # Ollama uses plain text prompt, not ChatPromptTemplate
            raw_prompt = SYSTEM_PROMPT.format(context=context) + "\n\n" + USER_PROMPT.format(
                contract_id           = str(row.get('contract_id', '')),
                buyer_name            = str(row.get('buyer_name', 'Unknown')),
                title                 = str(row.get('title', ''))[:300],
                description           = desc,
                winner_concentration  = int(row.get('winner_concentration', 0)),
                copy_paste_description= int(row.get('copy_paste_description', 0)),
                iso_forest_flag       = int(row.get('iso_forest_flag', 0)),
                iso_forest_score      = round(float(row.get('iso_forest_score', 0)), 3),
                buyer_contracts_count = int(row.get('buyer_contracts_count', 0)),
            )
            import re as _re
            raw_output = llm.invoke(raw_prompt)
            match = _re.search(r'\{.*\}', raw_output, _re.DOTALL)
            if not match:
                raise ValueError("No JSON found in LLM response")
            result = json.loads(match.group())
        else:
            # OpenAI / ChatOpenAI
            chain  = prompt_template | llm | JsonOutputParser()
            result = chain.invoke({
                'context'               : context,
                'contract_id'           : str(row.get('contract_id', '')),
                'buyer_name'            : str(row.get('buyer_name', 'Unknown')),
                'title'                 : str(row.get('title', ''))[:300],
                'description'           : desc,
                'winner_concentration'  : int(row.get('winner_concentration', 0)),
                'copy_paste_description': int(row.get('copy_paste_description', 0)),
                'iso_forest_flag'       : int(row.get('iso_forest_flag', 0)),
                'iso_forest_score'      : round(float(row.get('iso_forest_score', 0)), 3),
                'buyer_contracts_count' : int(row.get('buyer_contracts_count', 0)),
            })

        report = AuditReport(**result)
        return report.model_dump()

    except Exception as e:
        print(f"  LLM call failed ({e}), using rule-based fallback.")
        return _rule_based_report(row)


print('run_rag_audit() defined.')
print(f'Active backend: {LLM_BACKEND}')


## 8. Run on Sample Contracts

We process 25 suspicious contracts for the RAGAS evaluation.
Increase `N_SAMPLES` to cover more cases — each call costs one LLM API request.

In [ ]:
N_SAMPLES = 25

# Pick a diverse sample: high anomaly score + different flag combinations
sample = (
    suspicious
    .sort_values('iso_forest_score', ascending=False)
    .drop_duplicates(subset='buyer_id')   # one per buyer for diversity
    .head(N_SAMPLES)
)

print(f'Running RAG audit on {len(sample)} contracts...')

audit_results = []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Auditing'):
    result = run_rag_audit(row)
    audit_results.append(result)

# Save raw results
out_path = os.path.join(DATA_DIR, 'rag_audit_samples.json')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(audit_results, f, indent=2, ensure_ascii=False)

print(f'Saved {len(audit_results)} audit reports -> rag_audit_samples.json')

# Show one example
successful = [r for r in audit_results if 'error' not in r]
if successful:
    print('\n=== SAMPLE AUDIT REPORT ===')
    print(json.dumps(successful[0], indent=2))

## 9. RAGAS Evaluation

RAGAS measures three properties of the RAG pipeline:

| Metric | What it checks |
|---|---|
| **Faithfulness** | Does the answer contain only claims supported by the retrieved context? |
| **Answer Relevance** | Does the answer actually address the question asked? |
| **Context Precision** | Are the retrieved chunks actually relevant to the question? |

A faithful score < 0.7 means the LLM is hallucinating — making claims not in the legal context.
We target faithfulness > 0.85 for the audit reports to be usable in practice.

In [ ]:
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from datasets import Dataset

# Build RAGAS evaluation dataset from our audit results
ragas_rows = []

for i, (result, (_, row)) in enumerate(zip(audit_results, sample.iterrows())):
    if 'error' in result:
        continue

    # Re-run retrieval to get the context that was used
    query = (
        f"{row.get('title', '')}. "
        f"Flags: winner_concentration={row.get('winner_concentration', 0)}, "
        f"copy_paste={row.get('copy_paste_description', 0)}."
    )
    docs = retriever.invoke(query)
    contexts = [doc.page_content for doc in docs]

    ragas_rows.append({
        'question' : f"Is contract {row['contract_id']} suspicious and why?",
        'answer'   : result.get('explanation', ''),
        'contexts' : contexts,
        'ground_truth': (
            f"Contract flagged by: winner_concentration={row.get('winner_concentration', 0)}, "
            f"copy_paste={row.get('copy_paste_description', 0)}, "
            f"iso_forest={row.get('iso_forest_flag', 0)}"
        )
    })

print(f'RAGAS evaluation dataset: {len(ragas_rows)} rows')

if ragas_rows:
    ragas_dataset = Dataset.from_list(ragas_rows)

    ragas_scores = ragas_evaluate(
        ragas_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision]
    )

    scores_dict = ragas_scores.to_pandas()[[
        'faithfulness', 'answer_relevancy', 'context_precision'
    ]].mean().round(4).to_dict()

    print('\n=== RAGAS SCORES (mean across all samples) ===')
    for metric, score in scores_dict.items():
        status = '✅' if score >= 0.75 else '⚠️'
        print(f'  {metric:<25} {score:.4f}  {status}')

    # Save
    with open(os.path.join(DATA_DIR, 'ragas_results.json'), 'w') as f:
        json.dump(scores_dict, f, indent=2)
    print('\nSaved: ragas_results.json')
else:
    print('No successful audit results — check API key and LLM connection.')

## 10. CLI Utility — `analyze_contract.py`

The cell below writes the standalone CLI script to the project root.
After running it once, the team can analyse any contract from the terminal:

```bash
python analyze_contract.py --id IE_abc123...
```

In [ ]:
cli_path = os.path.join('..', 'analyze_contract.py')

CLI_SCRIPT = '\n#!/usr/bin/env python\n"""\nanalyze_contract.py\nUsage: python analyze_contract.py --id <contract_id>\nRuns the full RAG audit pipeline on a single contract and prints the JSON report.\n"""\n\nimport argparse\nimport json\nimport os\nimport re\nimport pandas as pd\nfrom pathlib import Path\n\nDATA_DIR    = "data"\nEMBED_MODEL = "paraphrase-multilingual-mpnet-base-v2"\nLLM_BACKEND = os.getenv("LLM_BACKEND", "ollama")\nOLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "mistral")\n\n\ndef load_pipeline():\n    from langchain_community.embeddings import HuggingFaceEmbeddings\n    from langchain_community.vectorstores import FAISS\n    embedder    = HuggingFaceEmbeddings(model_name=EMBED_MODEL)\n    vectorstore = FAISS.load_local(\n        os.path.join(DATA_DIR, "faiss_legal_index"),\n        embedder,\n        allow_dangerous_deserialization=True,\n    )\n    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})\n\n    llm = None\n    if LLM_BACKEND == "ollama":\n        from langchain_community.llms import Ollama\n        llm = Ollama(model=OLLAMA_MODEL, temperature=0)\n    elif LLM_BACKEND == "openai":\n        from langchain_openai import ChatOpenAI\n        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0,\n                         api_key=os.getenv("OPENAI_API_KEY", ""))\n    return retriever, llm\n\n\n_FLAG_RULES = {\n    "winner_concentration"  : ("Buyer in top 5% by volume.", 35),\n    "copy_paste_description": ("Near-identical description detected.", 30),\n    "single_bid"            : ("Single tender received.", 25),\n    "short_tender_period"   : ("Tender period below 35-day minimum.", 20),\n}\n\n\ndef rule_based(row) -> dict:\n    score, factors = 0, []\n    for col, (factor, weight) in _FLAG_RULES.items():\n        if row.get(col) == 1:\n            score += weight; factors.append(factor)\n    score  = min(score, 97)\n    action = "ESCALATE" if score >= 65 else ("INVESTIGATE" if score >= 35 else "MONITOR")\n    return {"contract_id": row.get("contract_id",""), "risk_score": score,\n            "risk_factors": factors, "recommended_action": action,\n            "explanation": f"Risk score {score}/100. Action: {action}."}\n\n\ndef analyze(contract_id: str):\n    df = pd.read_csv(os.path.join(DATA_DIR, "contracts_ie_clean.csv"), low_memory=False)\n    match = df[df["contract_id"] == contract_id]\n    if match.empty:\n        print(f"Contract not found: {contract_id}")\n        return\n\n    row = match.iloc[0]\n\n    if LLM_BACKEND == "rule":\n        result = rule_based(row)\n        print(json.dumps(result, indent=2))\n        return\n\n    retriever, llm = load_pipeline()\n\n    query = f"{row.get(\'title\', \'\')}. Checking for procurement fraud."\n    docs  = retriever.invoke(query)\n    context = "\\n\\n".join(\n        f"[{doc.metadata[\'source\']}]\\n{doc.page_content}" for doc in docs\n    )\n\n    prompt = (\n        f"You are an EU procurement auditor.\\n"\n        f"LEGAL CONTEXT:\\n{context}\\n\\n"\n        f"CONTRACT: {row.get(\'contract_id\',\'\')} | {row.get(\'buyer_name\',\'\')} | "\n        f"{str(row.get(\'title\',\'\'))[:200]}\\n"\n        f"Flags: winner_concentration={row.get(\'winner_concentration\',0)}, "\n        f"copy_paste={row.get(\'copy_paste_description\',0)}\\n\\n"\n        f"Respond ONLY with JSON: "\n        f"{{risk_score, risk_factors, regulation_references, explanation, recommended_action}}"\n    )\n\n    try:\n        raw = llm.invoke(prompt)\n        m   = re.search(r\'\\{.*\\}\', raw, re.DOTALL)\n        result = json.loads(m.group()) if m else rule_based(row)\n    except Exception:\n        result = rule_based(row)\n\n    result["contract_id"] = contract_id\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Audit a procurement contract for fraud risk.")\n    parser.add_argument("--id", required=True, help="Contract persistent ID")\n    args = parser.parse_args()\n    analyze(args.id)\n'

with open(cli_path, 'w', encoding='utf-8') as f:
    f.write(CLI_SCRIPT.strip())

print(f'Written: {os.path.abspath(cli_path)}')
print('Run with: python analyze_contract.py --id <contract_id>')
print(f'Active LLM backend: {LLM_BACKEND}')


## 11. Deliverables Summary

In [ ]:
deliverables = [
    ('rag_audit_samples.json',   'Structured audit reports for 25 suspicious contracts'),
    ('ragas_results.json',       'RAGAS scores: faithfulness, answer_relevancy, context_precision'),
    ('faiss_legal_index/',       'FAISS vector index of legal document chunks'),
    ('../analyze_contract.py',   'CLI tool: python analyze_contract.py --id <contract_id>'),
]

print(f'{"File":<35} {"Size":>10}  Description')
print('-' * 85)
for fname, desc in deliverables:
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        fpath = fname  # try relative path for CLI script

    if os.path.exists(fpath):
        size   = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
        sstr   = f'{size/1024:.0f} KB' if size > 0 else 'dir'
        status = 'OK'
    else:
        sstr   = '--'
        status = 'MISSING'
    print(f'{fname:<35} {sstr:>10}  [{status}] {desc}')